# Notebook: Tratamento da base CadÚnico

**id_rastreabilidade:** RTB_002  
**Versão:** v02  
**Data de criação:** 25/03/2026  
**Última atualização:** 08/05/2026  
**Responsável:** Ailton Vendramini  

---

## 🎯 Finalidade
Realizar a leitura, validação estrutural e tratamento técnico da base bruta do CadÚnico, preparando os dados para a camada limpa do projeto IVS-H.

---

## 📥 Fonte de Dados
**Fonte:** CadÚnico  
**Camada de entrada:** `01_bruto`  
**Camada de saída:** `02_limpo`  
**Período de referência da execução:** `2025_12`  
**Período de comparação:** —  
**Tipo de recorte temporal:** corte transversal  

**Arquivo de entrada:**  
`C:\\Users\\ailto\\Atlas-Social-de-Hortolandia\\dados\\cadunico\\01_bruto\\2025_12\\cadunico.csv`

**Arquivo de saída:**  
`C:\\Users\\ailto\\Atlas-Social-de-Hortolandia\\dados\\cadunico\\02_limpo\\2025_12\\cadunico_limpo.csv`

---

## 🧱 Base Conceitual
- `docs/metodologia_ivsh.md`
- `docs/regras_negocio.md`
- `docs/padrao_nomenclatura.md`

---

## ⚙️ Escopo técnico deste notebook

**Este notebook deve:**
1. Importações
2. Leitura da base bruta
3. Conferência de shape e colunas
4. Tipos de dados — diagnóstico e correção
5. Nulos — diagnóstico e tratamento
6. Datas — conversão para datetime
7. Criação de `idade`
8. Criação de `faixa_etaria`
9. Padronização textual mínima
10. Remoção de inconsistências gritantes
11. Exportação da base limpa

**Este notebook não deve:**
- calcular o IVS-H final
- misturar dados de outras fontes
- alterar diretamente arquivos da camada `01_bruto`

---

## 📊 Outputs esperados

| tipo_output | descrição | destino |
|---|---|---|
| exploratorio | validação inicial da base | não salvar |
| analitico | diagnósticos intermediários | `outputs/tabelas/cadunico/` |
| operacional | base limpa para próxima etapa | `dados/cadunico/02_limpo/2025_12/` |

---

## 🧠 Observações
- O arquivo bruto deve permanecer imutável.
- A data de referência para cálculo de idade é 31/12/2025 — data de corte do CadÚnico.
- Alterações estruturais relevantes devem ser registradas em nota técnica futura.

---
## 1. Importações

In [ ]:
import pandas as pd
import numpy as np
import os
from datetime import date

---
## 2. Leitura da base bruta

Parâmetros do `pd.read_csv`:
- `sep=';'` — CadÚnico usa ponto e vírgula como separador
- `encoding='latin1'` — evita erro com caracteres acentuados
- `low_memory=False` — lê tudo de uma vez e decide os tipos depois

In [ ]:
PERIODO_REFERENCIA = '2025_12'
DATA_REFERENCIA = date(2025, 12, 31)  # data de corte do CadÚnico

caminho_entrada = r'C:\Users\ailto\Atlas-Social-de-Hortolandia\dados\cadunico\01_bruto\2025_12\cadunico.csv'

df = pd.read_csv(
    caminho_entrada,
    sep=';',
    encoding='latin1',
    low_memory=False
)

print(f'Base carregada: {df.shape[0]} linhas, {df.shape[1]} colunas')

---
## 3. Conferência de shape e colunas

In [ ]:
# shape esperado: 72424 linhas, 211 colunas
print(f'Linhas  : {df.shape[0]}')
print(f'Colunas : {df.shape[1]}')

In [ ]:
# primeiras linhas para inspeção visual
df.head(3)

In [ ]:
# lista completa de colunas
print(df.columns.tolist())

---
## 4. Tipos de dados — diagnóstico e correção

Verificamos quantas colunas existem por tipo e identificamos as que precisam de ajuste.

In [ ]:
# distribuição de tipos
print(df.dtypes.value_counts())

In [ ]:
# colunas de renda devem ser numéricas
colunas_renda = [
    'd.vlr_renda_media_fam',
    'd.vlr_renda_total_fam'
]

for col in colunas_renda:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        print(f'{col} → {df[col].dtype}')

---
## 5. Nulos — diagnóstico e tratamento

No CadÚnico, o valor `999` indica ausência de resposta ou campo não aplicável.
Substituímos por `NaN` para não contaminar cálculos.

In [ ]:
# volume total de nulos por coluna (apenas as que têm)
nulos = df.isnull().sum()
nulos_presentes = nulos[nulos > 0].sort_values(ascending=False)
print(f'Colunas com nulos: {len(nulos_presentes)}')
nulos_presentes.head(20)

In [ ]:
# substituição de 999 por NaN nas colunas numéricas relevantes
colunas_999 = [
    'p.cod_sabe_ler_escrever_memb',
    'p.cod_grau_instr_pes',
    'p.cod_sexo_pessoa',
    'p.ind_frequenta_escola_memb'
]

for col in colunas_999:
    if col in df.columns:
        antes = (df[col] == 999).sum()
        df[col] = df[col].replace(999, np.nan)
        print(f'{col}: {antes} valores 999 → NaN')

---
## 6. Datas — conversão para datetime

Convertemos as colunas de data para o tipo `datetime`, permitindo operações de diferença temporal.

In [ ]:
colunas_data = [
    'p.dat_nasc_pes',
    'd.dat_cadastramento_fam',
    'd.dat_atual_fam',
    'd.dta_entrevista_fam'
]

for col in colunas_data:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], format='%d/%m/%Y', errors='coerce')
        invalidas = df[col].isnull().sum()
        print(f'{col} → datetime | datas inválidas: {invalidas}')

---
## 7. Criação de `idade`

A idade é calculada em relação à DATA_REFERENCIA (31/12/2025), não à data de hoje.
Isso garante que o cálculo seja reproduzível e rastreável ao período do CadÚnico.

In [ ]:
referencia = pd.Timestamp(DATA_REFERENCIA)

df['idade'] = (
    (referencia - df['p.dat_nasc_pes']).dt.days // 365
)

print(f'Coluna idade criada')
print(f'Mínima : {df["idade"].min()}')
print(f'Máxima : {df["idade"].max()}')
print(f'Nulos  : {df["idade"].isnull().sum()}')

---
## 8. Criação de `faixa_etaria`

Faixas definidas conforme critérios do IVS-H e políticas socioassistenciais.

In [ ]:
def classificar_faixa(idade):
    if pd.isna(idade) or idade < 0:
        return 'invalida'
    elif idade < 6:
        return '00_05'
    elif idade < 15:
        return '06_14'
    elif idade < 18:
        return '15_17'
    elif idade < 30:
        return '18_29'
    elif idade < 60:
        return '30_59'
    else:
        return '60_mais'

df['faixa_etaria'] = df['idade'].apply(classificar_faixa)

print(df['faixa_etaria'].value_counts())

---
## 9. Padronização textual mínima

Apenas o essencial: remoção de espaços e padronização de maiúsculas.
Matching territorial completo ocorre no notebook de análise.

In [ ]:
colunas_texto = [
    'd.nom_localidade_fam',
    'd.nom_logradouro_fam'
]

for col in colunas_texto:
    if col in df.columns:
        df[col] = df[col].str.strip().str.upper()
        print(f'{col} → padronizado')

---
## 10. Remoção de inconsistências gritantes

Apenas inconsistências objetivas e indefensáveis.
Casos ambíguos são documentados, não removidos.

In [ ]:
# idades inválidas: negativas ou acima de 120 anos
mask_idade_invalida = (df['idade'] < 0) | (df['idade'] > 120)
print(f'Registros com idade inválida: {mask_idade_invalida.sum()}')

In [ ]:
# renda absurda: acima de 100 salários mínimos (R$ 150.400 em dez/2025)
LIMITE_RENDA = 150400

if 'd.vlr_renda_media_fam' in df.columns:
    mask_renda_absurda = df['d.vlr_renda_media_fam'] > LIMITE_RENDA
    print(f'Registros com renda absurda (> R${LIMITE_RENDA}): {mask_renda_absurda.sum()}')

In [ ]:
# datas de nascimento futuras (nascimento após DATA_REFERENCIA)
if 'p.dat_nasc_pes' in df.columns:
    mask_nasc_futuro = df['p.dat_nasc_pes'] > referencia
    print(f'Nascimentos futuros: {mask_nasc_futuro.sum()}')

---
## 11. Exportação da base limpa

A base limpa é salva em `02_limpo` com encoding UTF-8.
O arquivo bruto em `01_bruto` permanece imutável.

In [ ]:
caminho_saida = r'C:\Users\ailto\Atlas-Social-de-Hortolandia\dados\cadunico\02_limpo\2025_12\cadunico_limpo.csv'

# garante que o diretório existe
os.makedirs(os.path.dirname(caminho_saida), exist_ok=True)

# exporta
df.to_csv(caminho_saida, sep=';', encoding='utf-8', index=False)

print(f'Base limpa exportada com sucesso')
print(f'Linhas  : {df.shape[0]}')
print(f'Colunas : {df.shape[1]}')
print(f'Destino : {caminho_saida}')